In [26]:
import pandas as pd
import numpy as np

In [27]:
df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\5번. 산업별 데이터\M19_도매_소매업.parquet')

In [28]:
df.isna().sum()

사업자등록번호        0
회계년도           0
회사명            0
종업원            0
설립일           35
              ..
영업이익률변화        0
부채비율변화         0
유동비율변화         0
부실라벨_ICR3년     0
M코드            0
Length: 109, dtype: int64

# 1. 분모가 <=0 일때 NaN값을 99th, 1st로 도출

train, test 데이터 나누기

Train 데이터의 99,1분위수를 기반으로 Train, Test의 NaN값을 대체함

In [29]:
import pandas as pd
import numpy as np

ID    = "사업자등록번호"
YEAR  = "회계년도"
LABEL = "부실라벨_ICR3년"   # ← 추가

# ═══════════════════════════════════════════════════════
# 대체값 그룹 정의
# ═══════════════════════════════════════════════════════

cols_99 = [
    "부채비율", "장기부채비율", "순차입금비율", "금융부채비율",
    "차입금의존도", "장기부채의존도", "총부채비율",
    "비유동비율", "비유동장기적합률", "유형자산부채비율",
    "유동비율", "당좌비율_추정", "현금비율",
    "현금전환주기_CCC",
    "매출채권회수기간", "재고자산보유기간", "매입채무지급기간",
    "ROE", "현금ROE", "유보율", "자본잠식률",
    "매출채권회전율", "재고자산회전율", "매입채무회전율",
    "영업CF_유동부채", "영업CF_총부채",
    "매출원가율", "판관비율", "금융비용부담률", "감가상각비율",
]

cols_1 = [
    "ROA", "ROIC", "총자본영업이익률", "현금ROA",
    "영업이익률", "순이익률", "EBITDA마진", "영업현금흐름비율",
    "매출총이익률",
    "총자산회전율", "유동자산회전율", "비유동자산회전율",
    "유형자산회전율", "투하자본회전율",
    "자기자본회전율", "순운전자본회전율",
    "자기자본비율", "순운전자본비율", "FCF_총자산",
    "유형자산비율",
]

cols_fixed = {
    "이자보상배율": 999,
}

# ═══════════════════════════════════════════════════════
# Train / Test 분리
# ═══════════════════════════════════════════════════════

train = df[df[YEAR] <= 2021].copy()
test  = df[df[YEAR] >= 2022].copy()

print(f"Train: {len(train):,}행 ({train[YEAR].min()}~{train[YEAR].max()})")
print(f"Test : {len(test):,}행  ({test[YEAR].min()}~{test[YEAR].max()})")

# ═══════════════════════════════════════════════════════
# STEP 1. 99th percentile 대체 (연도 × 라벨 그룹)
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 STEP 1. 99th percentile 대체 (연도 × 라벨 그룹)")
print("=" * 60)

# train 전체 기준 global p99 (fallback용)
train_p99_global = train[
    [c for c in cols_99 if c in train.columns]
].quantile(0.99)

# train 라벨별 global p99 (test fallback용)
train_p99_by_label = train.groupby(LABEL)[
    [c for c in cols_99 if c in train.columns]
].quantile(0.99)
# → index: label값(0 or 1), columns: 피처명

for col in cols_99:
    if col not in train.columns:
        print(f"  ⚠️  {col} 컬럼 없음 — 스킵")
        continue

    train_nan = train[col].isnull().sum()

    # ── Train: (연도 × 라벨) p99 → 라벨별 global p99 → 전체 global p99 순으로 fallback
    p99_by_year_label = train.groupby([YEAR, LABEL])[col].transform(
        lambda x: x.quantile(0.99)
    )
    train[col] = train[col].fillna(p99_by_year_label)

    # 라벨별 global p99 fallback
    label_p99_map = train_p99_by_label[col]          # Series: label → p99
    train[col] = train[col].fillna(train[LABEL].map(label_p99_map))

    # 전체 global p99 fallback (라벨 정보 없는 경우 대비)
    train[col] = train[col].fillna(train_p99_global[col])

    # ── Test: 라벨별 train p99 → 전체 train p99 순으로 fallback
    test_nan = test[col].isnull().sum()
    test[col] = test[col].fillna(test[LABEL].map(label_p99_map))
    test[col] = test[col].fillna(train_p99_global[col])

    print(
        f"  {col:<25} | "
        f"Train {train_nan:>5,} → {train[col].isnull().sum()} | "
        f"Test {test_nan:>5,} → {test[col].isnull().sum()}"
    )

# ═══════════════════════════════════════════════════════
# STEP 2. 1st percentile 대체 (연도 × 라벨 그룹)
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 STEP 2. 1st percentile 대체 (연도 × 라벨 그룹)")
print("=" * 60)

# train 전체 기준 global p01 (fallback용)
train_p01_global = train[
    [c for c in cols_1 if c in train.columns]
].quantile(0.01)

# train 라벨별 global p01 (test fallback용)
train_p01_by_label = train.groupby(LABEL)[
    [c for c in cols_1 if c in train.columns]
].quantile(0.01)

for col in cols_1:
    if col not in train.columns:
        print(f"  ⚠️  {col} 컬럼 없음 — 스킵")
        continue

    train_nan = train[col].isnull().sum()

    # ── Train: (연도 × 라벨) p01 → 라벨별 global p01 → 전체 global p01 순으로 fallback
    p01_by_year_label = train.groupby([YEAR, LABEL])[col].transform(
        lambda x: x.quantile(0.01)
    )
    train[col] = train[col].fillna(p01_by_year_label)

    label_p01_map = train_p01_by_label[col]
    train[col] = train[col].fillna(train[LABEL].map(label_p01_map))
    train[col] = train[col].fillna(train_p01_global[col])

    # ── Test: 라벨별 train p01 → 전체 train p01 순으로 fallback
    test_nan = test[col].isnull().sum()
    test[col] = test[col].fillna(test[LABEL].map(label_p01_map))
    test[col] = test[col].fillna(train_p01_global[col])

    print(
        f"  {col:<25} | "
        f"Train {train_nan:>5,} → {train[col].isnull().sum()} | "
        f"Test {test_nan:>5,} → {test[col].isnull().sum()}"
    )

# ═══════════════════════════════════════════════════════
# STEP 3. 고정값 대체
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 STEP 3. 고정값 대체")
print("=" * 60)

INT_EXP = "이자비용(요약)(백만원)"

for col, fixed_val in cols_fixed.items():
    if col not in train.columns:
        print(f"  ⚠️  {col} 컬럼 없음 — 스킵")
        continue

    # ── Train
    is_zero_train  = (train[INT_EXP].fillna(0) == 0) & train[col].isnull()
    is_other_train = (train[INT_EXP].fillna(0) != 0) & train[col].isnull()
    train.loc[is_zero_train, col] = fixed_val

    if is_other_train.sum() > 0:
        # 이자≠0 케이스도 (연도 × 라벨) 기준 p99로 대체
        p99_by_year_label = train.groupby([YEAR, LABEL])[col].transform(
            lambda x: x.quantile(0.99)
        )
        train.loc[is_other_train, col] = p99_by_year_label[is_other_train]

        # fallback: 라벨별 global p99
        still_null = is_other_train & train[col].isnull()
        if still_null.sum() > 0:
            label_p99_map = train_p99_by_label[col] if col in train_p99_by_label.columns else None
            if label_p99_map is not None:
                train.loc[still_null, col] = train.loc[still_null, LABEL].map(label_p99_map)
            train[col] = train[col].fillna(train_p99_global.get(col, fixed_val))

    # ── Test
    is_zero_test  = (test[INT_EXP].fillna(0) == 0) & test[col].isnull()
    is_other_test = (test[INT_EXP].fillna(0) != 0) & test[col].isnull()
    test.loc[is_zero_test, col] = fixed_val

    if is_other_test.sum() > 0:
        label_p99_map = train_p99_by_label[col] if col in train_p99_by_label.columns else None
        if label_p99_map is not None:
            test.loc[is_other_test, col] = test.loc[is_other_test, LABEL].map(label_p99_map)
        test.loc[test[col].isnull() & is_other_test, col] = train_p99_global.get(col, fixed_val)

    print(f"  {col:<25}")
    print(f"    Train | 이자=0: {is_zero_train.sum():,}개 → {fixed_val} / 이자≠0: {is_other_train.sum():,}개 → (연도×라벨) p99")
    print(f"    Test  | 이자=0: {is_zero_test.sum():,}개  → {fixed_val} / 이자≠0: {is_other_test.sum():,}개 → 라벨별 train p99")

# ═══════════════════════════════════════════════════════
# STEP 4. 잔여 NaN 확인
# ═══════════════════════════════════════════════════════
print("\n" + "=" * 60)
print("📊 STEP 4. 잔여 NaN 확인")
print("=" * 60)

target_cols = [c for c in cols_99 + cols_1 + list(cols_fixed.keys()) if c in train.columns]

for label, subset in [("Train", train), ("Test", test)]:
    remaining = subset[target_cols].isnull().sum()
    remaining = remaining[remaining > 0]
    if len(remaining) == 0:
        print(f"  ✅ {label} 잔여 NaN 없음")
    else:
        print(f"  ⚠️  {label} 잔여 NaN 존재")
        print(remaining.to_string())

# ═══════════════════════════════════════════════════════
# STEP 5. 재결합
# ═══════════════════════════════════════════════════════

df = pd.concat([train, test]).sort_index()

print("\n" + "=" * 60)
print("✅ 최종 NaN 현황")
print("=" * 60)
print(f"  Train 잔여 NaN : {train[target_cols].isnull().sum().sum():,}개")
print(f"  Test  잔여 NaN : {test[target_cols].isnull().sum().sum():,}개")
print(f"  전체  잔여 NaN : {df.isnull().sum().sum():,}개")

Train: 32,169행 (2010~2021)
Test : 11,797행  (2022~2024)

📊 STEP 1. 99th percentile 대체 (연도 × 라벨 그룹)
  부채비율                      | Train 1,505 → 0 | Test   428 → 0
  장기부채비율                    | Train 1,505 → 0 | Test   428 → 0
  순차입금비율                    | Train 1,505 → 0 | Test   428 → 0
  금융부채비율                    | Train 1,505 → 0 | Test   428 → 0
  차입금의존도                    | Train     2 → 0 | Test     0 → 0
  장기부채의존도                   | Train     2 → 0 | Test     0 → 0
  총부채비율                     | Train     2 → 0 | Test     0 → 0
  비유동비율                     | Train 1,505 → 0 | Test   428 → 0
  비유동장기적합률                  | Train   866 → 0 | Test   254 → 0
  유형자산부채비율                  | Train    12 → 0 | Test     4 → 0
  유동비율                      | Train    23 → 0 | Test     4 → 0
  당좌비율_추정                   | Train    23 → 0 | Test     4 → 0
  현금비율                      | Train    23 → 0 | Test     4 → 0
  현금전환주기_CCC                | Train 8,362 → 0 | Test 1,720 → 0
  매출채권회수기간          

In [30]:
# 매출액 음수 케이스 확인
매출액_col = "매출액(요약)(백만원)"

total = len(train)
neg_sales = (train[매출액_col] < 0).sum()
zero_sales = (train[매출액_col] == 0).sum()
pos_sales = (train[매출액_col] > 0).sum()

print("=" * 50)
print("📊 매출액 부호 분포 (Train 기준)")
print("=" * 50)
print(f"  양수 : {pos_sales:>7,}개 ({pos_sales/total*100:.2f}%)")
print(f"  0    : {zero_sales:>7,}개 ({zero_sales/total*100:.2f}%)")
print(f"  음수 : {neg_sales:>7,}개 ({neg_sales/total*100:.2f}%)")
print(f"  합계 : {total:>7,}개")

# 음수인 경우 자기자본회전율, 순운전자본회전율 극단값 확인
if neg_sales > 0:
    neg_mask = train[매출액_col] < 0
    for col in ["자기자본회전율", "순운전자본회전율"]:
        if col in train.columns:
            subset = train.loc[neg_mask, col].dropna()
            print(f"\n  [{col}] 매출액 음수 케이스")
            print(f"    건수  : {len(subset):,}개")
            print(f"    min   : {subset.min():.2f}")
            print(f"    max   : {subset.max():.2f}")
            print(f"    1st   : {subset.quantile(0.01):.2f}")
            print(f"    99th  : {subset.quantile(0.99):.2f}")

📊 매출액 부호 분포 (Train 기준)
  양수 :  31,792개 (98.83%)
  0    :     376개 (1.17%)
  음수 :       0개 (0.00%)
  합계 :  32,169개


In [31]:
print(df.isnull().sum().to_string())

사업자등록번호                       0
회계년도                          0
회사명                           0
종업원                           0
설립일                          35
금감원등록번호                       0
외부감사기관                        0
통계청 한국표준산업분류 코드 11차(대분류)      0
통계청 한국표준산업분류 11차(중분류)         0
자산총계(요약)(백만원)                 1
유동자산(요약)(백만원)                 1
현금 및 현금성자산(요약)(백만원)           1
매출채권(요약)(백만원)                 1
재고자산(요약)(백만원)                 1
비유동자산(요약)(백만원)                1
유형자산(요약)(백만원)                 1
부채총계(요약)(백만원)                 1
유동부채(요약)(백만원)                 1
매입채무(요약)(백만원)                 1
단기차입금(요약)(백만원)                1
유동성장기부채(요약)(백만원)              1
사채(요약)(백만원)                   1
장기차입금(요약)(백만원)                1
비유동부채(요약)(백만원)                1
자본총계(요약)(백만원)                 1
자본금(요약)(백만원)                  1
자본잉여금(요약)(백만원)                1
이익잉여금(요약)(백만원)                1
매출액(요약)(백만원)                  1
매출원가(요약)(백만원)                 1
매출총이익(요약)(백만원)                1
판매비와 관리비

In [32]:
print(train.isnull().sum().to_string())

사업자등록번호                       0
회계년도                          0
회사명                           0
종업원                           0
설립일                          32
금감원등록번호                       0
외부감사기관                        0
통계청 한국표준산업분류 코드 11차(대분류)      0
통계청 한국표준산업분류 11차(중분류)         0
자산총계(요약)(백만원)                 1
유동자산(요약)(백만원)                 1
현금 및 현금성자산(요약)(백만원)           1
매출채권(요약)(백만원)                 1
재고자산(요약)(백만원)                 1
비유동자산(요약)(백만원)                1
유형자산(요약)(백만원)                 1
부채총계(요약)(백만원)                 1
유동부채(요약)(백만원)                 1
매입채무(요약)(백만원)                 1
단기차입금(요약)(백만원)                1
유동성장기부채(요약)(백만원)              1
사채(요약)(백만원)                   1
장기차입금(요약)(백만원)                1
비유동부채(요약)(백만원)                1
자본총계(요약)(백만원)                 1
자본금(요약)(백만원)                  1
자본잉여금(요약)(백만원)                1
이익잉여금(요약)(백만원)                1
매출액(요약)(백만원)                  1
매출원가(요약)(백만원)                 1
매출총이익(요약)(백만원)                1
판매비와 관리비

In [33]:
print(test.isnull().sum().to_string())

사업자등록번호                      0
회계년도                         0
회사명                          0
종업원                          0
설립일                          3
금감원등록번호                      0
외부감사기관                       0
통계청 한국표준산업분류 코드 11차(대분류)     0
통계청 한국표준산업분류 11차(중분류)        0
자산총계(요약)(백만원)                0
유동자산(요약)(백만원)                0
현금 및 현금성자산(요약)(백만원)          0
매출채권(요약)(백만원)                0
재고자산(요약)(백만원)                0
비유동자산(요약)(백만원)               0
유형자산(요약)(백만원)                0
부채총계(요약)(백만원)                0
유동부채(요약)(백만원)                0
매입채무(요약)(백만원)                0
단기차입금(요약)(백만원)               0
유동성장기부채(요약)(백만원)             0
사채(요약)(백만원)                  0
장기차입금(요약)(백만원)               0
비유동부채(요약)(백만원)               0
자본총계(요약)(백만원)                0
자본금(요약)(백만원)                 0
자본잉여금(요약)(백만원)               0
이익잉여금(요약)(백만원)               0
매출액(요약)(백만원)                 0
매출원가(요약)(백만원)                0
매출총이익(요약)(백만원)               0
판매비와 관리비(요약)(백만원)            0
영업이익(요약)

# 2. 회계년도가 2010,2011년 데이터 삭제

In [34]:
df = df[~df['회계년도'].isin([2010, 2011])]

In [35]:
print(df.isnull().sum().to_string())

사업자등록번호                       0
회계년도                          0
회사명                           0
종업원                           0
설립일                          27
금감원등록번호                       0
외부감사기관                        0
통계청 한국표준산업분류 코드 11차(대분류)      0
통계청 한국표준산업분류 11차(중분류)         0
자산총계(요약)(백만원)                 0
유동자산(요약)(백만원)                 0
현금 및 현금성자산(요약)(백만원)           0
매출채권(요약)(백만원)                 0
재고자산(요약)(백만원)                 0
비유동자산(요약)(백만원)                0
유형자산(요약)(백만원)                 0
부채총계(요약)(백만원)                 0
유동부채(요약)(백만원)                 0
매입채무(요약)(백만원)                 0
단기차입금(요약)(백만원)                0
유동성장기부채(요약)(백만원)              0
사채(요약)(백만원)                   0
장기차입금(요약)(백만원)                0
비유동부채(요약)(백만원)                0
자본총계(요약)(백만원)                 0
자본금(요약)(백만원)                  0
자본잉여금(요약)(백만원)                0
이익잉여금(요약)(백만원)                0
매출액(요약)(백만원)                  0
매출원가(요약)(백만원)                 0
매출총이익(요약)(백만원)                0
판매비와 관리비

중앙값 대체는 산업별로 나누고, train-test 데이터 나누고

In [36]:
df.to_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\6번.데이터채우기전\M19_도매_소매업.parquet')

# 비재무변수(외부감사기관, 설립일) 결측치 처리

In [37]:
import pandas as pd

df = pd.read_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\6번.데이터채우기전\M19_도매_소매업.parquet')

## 결측값인 설립일 데이터 Dart 로 가져오기

In [38]:
df.loc[df['설립일'].isna(), '회사명'].drop_duplicates()

16        (주)신화사
2838     동하산업(주)
10564    제동물산(주)
16542    (주)국제철강
Name: 회사명, dtype: str

In [14]:
import pandas as pd
import requests
import zipfile
import io
import xml.etree.ElementTree as ET
import re

# ============================================
# DART API KEY
# ============================================
API_KEY = "55f022874c5017a0ee5aa381c98c0f6ebc2a14ce"

# ============================================
# 회사명 정규화 함수
# ============================================
def normalize_company_name(name):

    if pd.isna(name):
        return ""

    name = str(name)

    # (주), ㈜ 제거
    name = re.sub(r'\(주\)|㈜', '', name)

    # 주식회사 제거
    name = name.replace("주식회사", "")

    # 공백 제거
    name = re.sub(r'\s+', '', name)

    return name.strip()

# ============================================
# DART 기업코드 다운로드
# ============================================
print("DART 기업코드 다운로드 중...")

url = (
    f"https://opendart.fss.or.kr/api/corpCode.xml"
    f"?crtfc_key={API_KEY}"
)

res = requests.get(url)

zf = zipfile.ZipFile(io.BytesIO(res.content))
xml_file = zf.open("CORPCODE.xml")

tree = ET.parse(xml_file)
root = tree.getroot()

corp_list = []

for corp in root.findall("list"):

    corp_list.append({
        "corp_code": corp.findtext("corp_code"),
        "corp_name": corp.findtext("corp_name")
    })

corp_df = pd.DataFrame(corp_list)

corp_df["corp_name_norm"] = (
    corp_df["corp_name"]
    .apply(normalize_company_name)
)

# ============================================
# 기업명 → corp_code 매핑
# ============================================
corp_code_map = dict(
    zip(
        corp_df["corp_name_norm"],
        corp_df["corp_code"]
    )
)

# ============================================
# 설립일 조회 함수
# ============================================
def get_establishment_date(corp_code):

    url = "https://opendart.fss.or.kr/api/company.json"

    params = {
        "crtfc_key": API_KEY,
        "corp_code": corp_code
    }

    try:

        r = requests.get(url, params=params)
        data = r.json()

        if data.get("status") == "000":
            return data.get("est_dt")

    except Exception:
        pass

    return None

# ============================================
# df 회사명 정규화
# ============================================
df["회사명_norm"] = (
    df["회사명"]
    .apply(normalize_company_name)
)

# ============================================
# 설립일 결측치 채우기
# ============================================
filled_count = 0
unmatched_companies = []

mask = df["설립일"].isna()

for idx in df[mask].index:

    company = df.loc[idx, "회사명"]
    company_norm = df.loc[idx, "회사명_norm"]

    corp_code = corp_code_map.get(company_norm)

    if corp_code is None:

        unmatched_companies.append(company)
        continue

    est_dt = get_establishment_date(corp_code)

    if est_dt:

        df.loc[idx, "설립일"] = est_dt
        filled_count += 1

        print(f"[채움] {company} → {est_dt}")

# ============================================
# 결과 확인
# ============================================
print("\n" + "=" * 50)
print(f"채워진 건수 : {filled_count}")
print(f"남은 결측치 : {df['설립일'].isna().sum()}")

print("\n[매핑 실패 기업]")
for company in sorted(set(unmatched_companies)):
    print(company)

# 임시 컬럼 제거
df.drop(columns=["회사명_norm"], inplace=True)

print("\n완료")

DART 기업코드 다운로드 중...

채워진 건수 : 0
남은 결측치 : 27

[매핑 실패 기업]

완료


In [15]:
missing_companies = (
    df.loc[df['설립일'].isna(), '회사명']
      .drop_duplicates()
      .sort_values()
)

print("결측 기업 수 :", len(missing_companies))
print(missing_companies.tolist())

결측 기업 수 : 4
['(주)국제철강', '(주)신화사', '동하산업(주)', '제동물산(주)']


In [16]:
print("채워진 기업 수")

filled_companies = (
    df.loc[df['설립일'].notna(), '회사명']
      .drop_duplicates()
)

print(len(filled_companies))

채워진 기업 수
6102


In [17]:
df["회사명_norm"] = df["회사명"].apply(normalize_company_name)

matched = df["회사명_norm"].isin(corp_code_map)

print("전체 기업:", df["회사명_norm"].nunique())
print("매핑 성공:", df.loc[matched, "회사명_norm"].nunique())
print("매핑 실패:", df.loc[~matched, "회사명_norm"].nunique())

전체 기업: 6068
매핑 성공: 6016
매핑 실패: 52


매핑은 잘 됐는데, Dart에 '설립일' 항목에 값이 없어서 값을 채울 수 없음. 

- 기존 설립일 값이 없는 건수 : 207
    - 설립일 값이 있어서 채워진 건수 : 28
    - 설립일 값이 없어서 채워지지 못한 건수 : 179

남은 결측값에 대해서는 이와 같이 결측치 대체할 것임

- 설립일

↓

- 설립연수 생성 (설립연수 = 회계년도 - 설립연도)

↓

- 산업별 중앙값 대체

↓

- 원본 설립일 컬럼 제거

## 결측값인 외부감사기관 데이터 채우기

In [18]:
df.loc[df['외부감사기관'].isna(), '회사명'].drop_duplicates()

Series([], Name: 회사명, dtype: str)

외부감사기관 데이터가 없는 결측 기업은 고작 4개임. 그래서 Unknown 으로 결측치 대체함

In [19]:
df["외부감사기관"] = df["외부감사기관"].fillna("Unknown")

In [20]:
df.isna().sum()

사업자등록번호        0
회계년도           0
회사명            0
종업원            0
설립일           27
              ..
부채비율변화         0
유동비율변화         0
부실라벨_ICR3년     0
M코드            0
회사명_norm       0
Length: 110, dtype: int64

In [21]:
df.to_parquet(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\데이터\6번.데이터채우기후\M19_도매_소매업.parquet')